# Previsão do valor do aluguel de imóveis no Brasil
**Disciplina:** Tópicos Avançados em Inteligência Artificial — Trabalho prático 1

**Problema:** estimar o **valor mensal do aluguel (R$)** de um imóvel residencial a partir das suas
características (cidade, área, quartos, banheiros, vagas, andar, condomínio, mobília e se aceita animais).

**Quem usa:** quem procura um imóvel (para saber se o preço anunciado é razoável) ou proprietários que
querem definir o valor do anúncio. **Saída:** aluguel estimado em R$/mês.

**Tipo de problema:** regressão.

**Alternativas comparadas:**
1. Regressão Linear Múltipla (com log do aluguel e log da área);
2. Árvore de Decisão para regressão (hiperparâmetros por validação cruzada);
3. Random Forest (conjunto de árvores) — modelo pesquisado além da aula, justificado na Seção 6.

**Dataset:** *Brazilian Houses to Rent* (Kaggle, usuário *rubenssjr*; também disponível no OpenML como
*Brazilian_houses*): 10.692 anúncios de 5 cidades, coletados de um site de imóveis em 2020.

> Execute as células **em ordem** (Ambiente de execução → Executar tudo).

## 1. Importação das bibliotecas

In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42                                   # semente fixa = resultados reproduzíveis
AZUL, LARANJA, VERDE = '#2a78d6', '#eb6834', '#1baf7a'
sns.set_theme(style='whitegrid')

# Registramos as versões: elas vão para o requirements.txt e precisam ser IGUAIS no deploy
print('Python      :', sys.version.split()[0])
print('pandas      :', pd.__version__)
print('numpy       :', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('joblib      :', joblib.__version__)

## 2. Pastas do projeto e Google Drive
Como nas aulas, os arquivos são salvos no Google Drive para não se perderem quando o Colab desconectar.
A estrutura criada é a mesma que será enviada ao GitHub no deploy:

```
Projeto_Aluguel/
├── dataset/
├── modelo/
└── requirements.txt
```
Se o notebook for executado fora do Colab, ele usa a pasta atual.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/Lab_IA/Projeto_Aluguel'
except ImportError:          # executando fora do Colab (ex.: Jupyter local)
    PASTA = '.'

os.makedirs(f'{PASTA}/dataset', exist_ok=True)
os.makedirs(f'{PASTA}/modelo', exist_ok=True)
print('Arquivos serão salvos em:', os.path.abspath(PASTA))

## 3. Obtenção do dataset
O CSV é baixado de um espelho público no GitHub (o Kaggle exige login para download direto) e uma
cópia é salva em `dataset/houses_to_rent_v2.csv`, garantindo que o projeto use sempre a mesma versão.

| Coluna original | Nome usado | Significado |
|---|---|---|
| `city` | `cidade` | São Paulo, Rio de Janeiro, Belo Horizonte, Porto Alegre ou Campinas |
| `area` | `area` | área do imóvel (m²) |
| `rooms` | `quartos` | nº de quartos |
| `bathroom` | `banheiros` | nº de banheiros |
| `parking spaces` | `vagas` | nº de vagas de garagem |
| `floor` | `andar` | andar (`-` = casa/térreo) |
| `animal` | `animal` | aceita animais (`acept` / `not acept`) |
| `furniture` | `mobiliado` | mobiliado (`furnished` / `not furnished`) |
| `hoa (R$)` | `condominio` | taxa de condomínio |
| **`rent amount (R$)`** | **`aluguel`** | **valor do aluguel — variável-alvo** |
| `property tax (R$)` | `iptu` | IPTU mensal |
| `fire insurance (R$)` | `seguro` | seguro incêndio |
| `total (R$)` | `total` | soma de aluguel + condomínio + IPTU + seguro |

In [ ]:
URL = 'https://raw.githubusercontent.com/marcos-s1/Aluguel/master/houses_to_rent_v2.csv'
ARQ_DATASET = f'{PASTA}/dataset/houses_to_rent_v2.csv'

if os.path.exists(ARQ_DATASET):
    bruto = pd.read_csv(ARQ_DATASET)
else:
    bruto = pd.read_csv(URL)
    bruto.to_csv(ARQ_DATASET, index=False)

# Renomeia as colunas para português (mais fácil de ler no código e no relatório)
dataset = bruto.copy()
dataset.columns = ['cidade', 'area', 'quartos', 'banheiros', 'vagas', 'andar', 'animal', 'mobiliado',
                   'condominio', 'aluguel', 'iptu', 'seguro', 'total']
print('Formato (linhas, colunas):', dataset.shape)
dataset.head()

## 4. Análise exploratória
### 4.1 Tipos, ausentes, duplicatas e valores estranhos

In [ ]:
dataset.info()
print('\nValores ausentes:', dataset.isnull().sum().sum())
print('Linhas duplicadas:', dataset.duplicated().sum())
print('\nValores da coluna andar:', sorted(dataset['andar'].unique(), key=lambda v: (len(v), v))[:10], '...')

In [ ]:
dataset.describe().round(1)

In [ ]:
# Os maiores valores de área, andar e condomínio — claramente erros de digitação
print(dataset.nlargest(5, 'area')[['cidade', 'area', 'quartos', 'aluguel']], '\n')
print(dataset[dataset['andar'] != '-'].assign(andar=lambda d: d['andar'].astype(int))
      .nlargest(3, 'andar')[['cidade', 'andar', 'aluguel']], '\n')
print(dataset.nlargest(5, 'condominio')[['cidade', 'area', 'condominio', 'aluguel']])

**Observações:** não há valores ausentes, mas existem **358 linhas duplicadas** (o mesmo anúncio repetido),
a coluna `andar` é texto (`-` para casas/térreo) e há valores impossíveis: área de 46.335 m² para um
imóvel de 1 quarto, andar 301 e condomínio de R$ 1,1 milhão. Esses casos serão tratados na Seção 5.

### 4.2 Vazamento de informação: `seguro` e `total`

In [ ]:
# O seguro incêndio e o total são CALCULADOS a partir do aluguel: usá-los seria "colar" a resposta.
print(dataset[['aluguel', 'seguro', 'total', 'condominio', 'iptu']].corr()['aluguel'].round(3))

O seguro incêndio tem correlação ~0,99 com o aluguel (é calculado como um percentual dele) e o total
**contém** o aluguel. Ambos são **removidos**. O IPTU também fica de fora: muitos anúncios têm valor zero
ou absurdo e o inquilino nem sempre sabe esse valor antes de pesquisar. Já o **condomínio** é mantido,
porque é uma taxa independente que aparece em qualquer anúncio.

### 4.3 Distribuição do aluguel e relação com os atributos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(dataset['aluguel'], bins=60, color=AZUL, edgecolor='white')
axes[0].set_title('Distribuição do aluguel')
axes[0].set_xlabel('Aluguel (R$)')
axes[0].set_ylabel('Nº de imóveis')
axes[1].hist(np.log(dataset['aluguel']), bins=60, color=AZUL, edgecolor='white')
axes[1].set_title('Distribuição do log do aluguel')
axes[1].set_xlabel('log(aluguel)')
plt.show()

O aluguel é muito assimétrico (poucos imóveis muito caros). Em escala logarítmica fica quase simétrico —
por isso a regressão linear será treinada com **log(aluguel)**.

In [ ]:
ordem = dataset.groupby('cidade')['aluguel'].median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=dataset, x='cidade', y='aluguel', order=ordem, color=AZUL, showfliers=False, ax=ax)
ax.set_title('Aluguel por cidade (sem outliers no gráfico)')
ax.set_xlabel('')
ax.set_ylabel('Aluguel (R$)')
plt.show()
print(dataset.groupby('cidade')['aluguel'].median().sort_values(ascending=False))

In [ ]:
amostra = dataset[dataset['area'] <= 1000]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
axes[0].scatter(amostra['area'], amostra['aluguel'], s=8, alpha=0.3, color=AZUL)
axes[0].set_xlabel('Área (m²)')
axes[0].set_ylabel('Aluguel (R$)')
axes[0].set_title('Aluguel x área')
sns.boxplot(data=dataset, x='mobiliado', y='aluguel', hue='mobiliado', palette=[LARANJA, AZUL],
            showfliers=False, legend=False, ax=axes[1])
axes[1].set_title('Aluguel x mobília')
axes[1].set_xlabel('')
plt.show()

In [ ]:
corr = dataset[dataset['area'] <= 1000][['area', 'quartos', 'banheiros', 'vagas', 'condominio', 'aluguel']].corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, ax=ax)
ax.set_title('Correlação entre os atributos numéricos')
plt.show()

**Principais achados:** São Paulo é a cidade mais cara e Campinas a mais barata; aluguel cresce com
área, banheiros e vagas (banheiros é o atributo mais correlacionado); imóveis mobiliados são mais caros.
A relação com a área não é linear: o preço por m² diminui em imóveis grandes — por isso usaremos
**log(área)** na regressão linear.

## 5. Preparação dos dados
### 5.1 Limpeza
Decisões (todas justificadas na Seção 4):
* remover as **duplicatas**;
* converter `andar`: `-` vira 0 e criamos a coluna `casa_terreo` (1 = casa ou térreo);
* remover **valores impossíveis**: área fora de 15–1.000 m², andar acima de 50, condomínio acima de R$ 20 mil
  (afetam menos de 1% das linhas);
* descartar `seguro`, `total` (vazamento) e `iptu`.

In [ ]:
dados = dataset.drop_duplicates().copy()
dados['andar'] = dados['andar'].replace('-', '0').astype(int)

antes = len(dados)
dados = dados[dados['area'].between(15, 1000) & (dados['andar'] <= 50) & (dados['condominio'] <= 20000)]
dados = dados.drop(columns=['seguro', 'total', 'iptu']).reset_index(drop=True)
print(f'{len(dataset)} linhas -> {antes} sem duplicatas -> {len(dados)} após remover valores impossíveis')

### 5.2 Atributos derivados
A **mesma função** é copiada no `app.py`, para a entrada do usuário ser processada igual ao treino.
* `casa_terreo` = 1 se andar = 0;
* `log_area` = logaritmo da área (relação mais linear com o log do aluguel).

In [ ]:
def criar_atributos(df):
    """Recebe um DataFrame com as colunas originais e devolve uma cópia com os atributos derivados."""
    df = df.copy()
    df['casa_terreo'] = (df['andar'] == 0).astype(int)
    df['log_area'] = np.log(df['area'])
    return df

dados = criar_atributos(dados)
dados.head()

### 5.3 Separação treino/teste
80% treino / 20% teste. O teste só é usado na avaliação final (Seção 7).

In [ ]:
X = dados.drop(columns='aluguel')
y = dados['aluguel']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print('Treino:', X_train.shape, '| Teste:', X_test.shape)

### 5.4 Pré-processamento
* Categóricas (`cidade`, `animal`, `mobiliado`) → `OneHotEncoder(drop='first')`;
* Numéricas → `StandardScaler`.

Tudo fica dentro de um `Pipeline`: o pré-processamento é ajustado **só com o treino** e vai junto no
arquivo salvo.

In [ ]:
CATEGORICAS = ['cidade', 'animal', 'mobiliado']
NUM_LINEAR = ['log_area', 'quartos', 'banheiros', 'vagas', 'andar', 'casa_terreo', 'condominio']
NUM_ARVORE = ['area', 'quartos', 'banheiros', 'vagas', 'andar', 'casa_terreo', 'condominio']

def criar_preprocessador(numericas):
    return ColumnTransformer([
        ('num', StandardScaler(), numericas),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAS),
    ], remainder='drop')

## 6. Treinamento e comparação dos modelos
Comparação por **validação cruzada (5 dobras) no treino**. Métrica principal: **MAE** em R$.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)

def avaliar_cv(modelo, nome):
    mae = -cross_val_score(modelo, X_train, y_train, cv=kfold, scoring='neg_mean_absolute_error')
    print(f'{nome:<32} MAE = R$ {mae.mean():7,.0f} (± {mae.std():,.0f})')
    return {'Modelo': nome, 'MAE (CV)': mae.mean()}

### 6.1 Modelo 1 — Regressão Linear com log
`TransformedTargetRegressor` treina a regressão com **log(aluguel)** e converte a previsão de volta para
reais automaticamente (com `exp`).

In [ ]:
modelo_lin = TransformedTargetRegressor(
    regressor=Pipeline([('prep', criar_preprocessador(NUM_LINEAR)), ('reg', LinearRegression())]),
    func=np.log, inverse_func=np.exp)
res_lin = avaliar_cv(modelo_lin, 'Regressão Linear (log)')

### 6.2 Modelo 2 — Árvore de Decisão (busca de hiperparâmetros)

In [ ]:
busca_arvore = GridSearchCV(
    Pipeline([('prep', criar_preprocessador(NUM_ARVORE)), ('reg', DecisionTreeRegressor(random_state=SEED))]),
    {'reg__max_depth': [4, 6, 8, 10, 12, None], 'reg__min_samples_leaf': [5, 10, 20, 40]},
    cv=kfold, scoring='neg_mean_absolute_error')
busca_arvore.fit(X_train, y_train)
print('Melhor configuração:', busca_arvore.best_params_)
modelo_arvore = busca_arvore.best_estimator_
res_arvore = avaliar_cv(modelo_arvore, 'Árvore de Decisão')

### 6.3 Modelo 3 — Random Forest
Uma única árvore é instável: pequenas mudanças nos dados mudam muito a árvore. A **Random Forest** treina
várias árvores em amostras diferentes dos dados e tira a **média** das previsões, reduzindo esse problema.
Limitamos `max_depth` e o nº de árvores para o arquivo do modelo continuar pequeno (deploy no GitHub).

In [ ]:
busca_rf = GridSearchCV(
    Pipeline([('prep', criar_preprocessador(NUM_ARVORE)),
              ('reg', RandomForestRegressor(n_estimators=100, max_depth=14, random_state=SEED, n_jobs=-1))]),
    {'reg__min_samples_leaf': [3, 5, 10]},
    cv=kfold, scoring='neg_mean_absolute_error')
busca_rf.fit(X_train, y_train)
print('Melhor configuração:', busca_rf.best_params_)
modelo_rf = busca_rf.best_estimator_
res_rf = avaliar_cv(modelo_rf, 'Random Forest')

### 6.4 Escolha do modelo (pelo MAE da validação cruzada — o teste não participa)

In [ ]:
comparacao_cv = pd.DataFrame([res_lin, res_arvore, res_rf]).set_index('Modelo').round(0)
display(comparacao_cv)
modelos = {'Regressão Linear (log)': modelo_lin, 'Árvore de Decisão': modelo_arvore, 'Random Forest': modelo_rf}
nome_escolhido = comparacao_cv['MAE (CV)'].idxmin()
print('Modelo escolhido para o deploy:', nome_escolhido)

## 7. Avaliação final no conjunto de teste

In [ ]:
linhas, previsoes = [], {}
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    previsoes[nome] = y_pred
    linhas.append({'Modelo': nome,
                   'MAE': mean_absolute_error(y_test, y_pred),
                   'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
                   'R²': r2_score(y_test, y_pred),
                   'Erro % mediano': np.median(np.abs(y_test - y_pred) / y_test) * 100})
resultado_teste = pd.DataFrame(linhas).set_index('Modelo')
display(resultado_teste.style.format({'MAE': 'R$ {:,.0f}', 'RMSE': 'R$ {:,.0f}', 'R²': '{:.3f}',
                                      'Erro % mediano': '{:.1f}%'}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
limite = [0, 20000]
for ax, (nome, y_pred), cor in zip(axes, previsoes.items(), [AZUL, LARANJA, VERDE]):
    ax.scatter(y_test, y_pred, s=8, alpha=0.35, color=cor)
    ax.plot(limite, limite, color='gray', linestyle='--', linewidth=1)
    ax.set_xlim(limite); ax.set_ylim(limite)
    ax.set_title(f"{nome}\nMAE = R$ {resultado_teste.loc[nome, 'MAE']:,.0f}", fontsize=10)
    ax.set_xlabel('Aluguel real (R$)')
axes[0].set_ylabel('Aluguel previsto (R$)')
plt.show()

### 7.1 Análise dos erros do modelo escolhido

In [ ]:
modelo_final = modelos[nome_escolhido]
analise = X_test.copy()
analise['real'] = y_test
analise['previsto'] = modelo_final.predict(X_test)
analise['erro_abs'] = (analise['real'] - analise['previsto']).abs()
analise['erro_%'] = analise['erro_abs'] / analise['real'] * 100

analise['faixa'] = pd.cut(analise['real'], [0, 2000, 5000, 10000, np.inf],
                          labels=['até 2 mil', '2–5 mil', '5–10 mil', 'acima de 10 mil'])
display(analise.groupby('cidade')[['erro_abs', 'erro_%']].median().round(1)
        .rename(columns={'erro_abs': 'erro mediano (R$)', 'erro_%': 'erro % mediano'}))
display(analise.groupby('faixa', observed=True)[['erro_abs', 'erro_%']].median().round(1)
        .rename(columns={'erro_abs': 'erro mediano (R$)', 'erro_%': 'erro % mediano'}))
print('Os 5 maiores erros:')
analise.nlargest(5, 'erro_abs')[['cidade', 'area', 'quartos', 'banheiros', 'vagas', 'mobiliado',
                                 'condominio', 'real', 'previsto']].round(0)

In [ ]:
# O que mais pesa na previsão (importância dos atributos nos modelos de árvore)
reg = modelo_rf.named_steps['reg']
nomes = modelo_rf.named_steps['prep'].get_feature_names_out()
imp = pd.Series(reg.feature_importances_, index=nomes).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp.index, imp.values, color=VERDE)
ax.set_title('Importância dos atributos — Random Forest')
plt.show()

### 7.2 Discussão (adapte com os números obtidos)
* Os três modelos têm erro parecido (R² em torno de 0,6): o preço depende muito de coisas que **não estão
  no dataset** — bairro, estado de conservação, vista, lazer do prédio. Dentro de São Paulo, por exemplo, o
  mesmo apartamento custa o dobro em bairros diferentes.
* A Random Forest tende a errar menos por combinar várias árvores; a regressão linear em log é a mais fácil
  de explicar (cada coeficiente vira um "percentual a mais").
* O erro **percentual** fica entre 20% e 30% na maioria das faixas, mas piora acima de R$ 10 mil: imóveis
  caros são poucos no dataset e a floresta tende a subestimá-los (veja os maiores erros).

**Limitações:** anúncios de 2020 (preços desatualizados — o modelo não considera inflação), apenas 5 cidades,
sem bairro, preço anunciado ≠ preço fechado em contrato.

## 8. Salvando o modelo e os arquivos do deploy

In [ ]:
pacote = {
    'modelo': modelo_final,
    'nome_modelo': nome_escolhido,
    'metricas_teste': resultado_teste.loc[nome_escolhido].round(3).to_dict(),
    'cidades': sorted(dados['cidade'].unique()),
    'faixas': {col: (float(X[col].min()), float(X[col].max()))
               for col in ['area', 'quartos', 'banheiros', 'vagas', 'andar', 'condominio']},
    'versao_sklearn': sklearn.__version__,
}
ARQ_MODELO = f'{PASTA}/modelo/modelo_aluguel.pkl'
joblib.dump(pacote, ARQ_MODELO, compress=3)        # compress=3 deixa o arquivo menor para o GitHub
print('Modelo salvo em:', ARQ_MODELO, f'({os.path.getsize(ARQ_MODELO) / 1e6:.1f} MB)')
print(pacote['nome_modelo'], pacote['metricas_teste'])

In [ ]:
# requirements.txt com as MESMAS versões deste ambiente.
# Importante: o .pkl só abre corretamente com a mesma versão do scikit-learn usada aqui.
requisitos = f"""streamlit==1.50.0
scikit-learn=={sklearn.__version__}
pandas=={pd.__version__}
numpy=={np.__version__}
joblib=={joblib.__version__}
"""
with open(f'{PASTA}/requirements.txt', 'w') as f:
    f.write(requisitos)
print(requisitos)
print('Python usado (escolha a MESMA versão principal no Streamlit Cloud):', sys.version.split()[0])

## 9. Teste de inferência (casos de teste do relatório)
Recarregamos o arquivo salvo (como o `app.py` faz) e prevemos casos que devem ser repetidos no app.

In [ ]:
modelo_carregado = joblib.load(ARQ_MODELO)['modelo']
casos = pd.DataFrame([
    {'caso': 'Válido 1 - kitnet SP',            'cidade': 'São Paulo',      'area': 35,  'quartos': 1, 'banheiros': 1, 'vagas': 0, 'andar': 5,  'animal': 'not acept', 'mobiliado': 'furnished',     'condominio': 450},
    {'caso': 'Válido 2 - apto família BH',      'cidade': 'Belo Horizonte', 'area': 90,  'quartos': 3, 'banheiros': 2, 'vagas': 2, 'andar': 3,  'animal': 'acept',     'mobiliado': 'not furnished', 'condominio': 500},
    {'caso': 'Válido 3 - casa Porto Alegre',    'cidade': 'Porto Alegre',   'area': 200, 'quartos': 4, 'banheiros': 3, 'vagas': 2, 'andar': 0,  'animal': 'acept',     'mobiliado': 'not furnished', 'condominio': 0},
    {'caso': 'Limítrofe - cobertura 900 m² RJ', 'cidade': 'Rio de Janeiro', 'area': 900, 'quartos': 6, 'banheiros': 7, 'vagas': 5, 'andar': 30, 'animal': 'acept',     'mobiliado': 'furnished',     'condominio': 9000},
])
casos['previsto (R$)'] = modelo_carregado.predict(criar_atributos(casos.drop(columns='caso'))).round(2)
casos

**Caso limítrofe:** imóveis de altíssimo padrão são raros no treino; árvores não conseguem prever acima do
maior valor visto nas folhas, então a estimativa tende a ficar **abaixo** do real. O caso inválido (campo
vazio, área zero) é testado no app.

## 10. Download dos arquivos
Os arquivos já estão no seu Google Drive. Se preferir baixá-los direto para o computador, execute a célula abaixo.

In [ ]:
try:
    from google.colab import files
    for arquivo in [ARQ_MODELO, ARQ_DATASET, f'{PASTA}/requirements.txt']:
        files.download(arquivo)
except ImportError:
    print('Fora do Colab: os arquivos estão em', os.path.abspath(PASTA))